# 03. Messaging Channels

Notebook `01-team-formation` showed how a roster comes together: `EntityRegistry`, `Entity`, `AuditLogger`. Notebook `02-task-distribution` covers the *patterns* for fanning work out across that roster — `Channel`, `MsgType`, `Priority`, who-talks-to-whom shape questions. This notebook is the wire layer. We open the box and look at the actual messages flowing between members: how they're constructed, where they land, how a recipient pulls them, and what makes the stream tamper-evident.

The package's name for this layer is `MessagingService` and its envelope is `Message`. Both live in `arcteam.messenger` / `arcteam.types`. They're built around a few hard design choices:

- **Pull-based**, not push. Recipients call `poll()` on a stream; the service never reaches into a runtime to deliver. This matches the "shared-nothing per agent" principle in `arcteam`'s build standards.
- **Append-only streams**, addressed by URI. `agent://name`, `channel://name`, `role://name`, `user://name`. The service routes a single `send()` to one or more streams, one append per target.
- **Cursor-driven receive**. Each `(stream, entity)` pair owns a `Cursor` recording last-acked `seq` and `byte_pos`. Polls read forward from the cursor; `ack()` advances it. Forward-only — backward acks are silently rejected.
- **Durable + replayable**. The backend (`MemoryBackend` here, `NatsBackend`/JetStream in production) keeps every message forever. Re-poll without ack and the same message comes back. Crash, restart, re-poll: same message.
- **Tamper-evident**. Every `send`, `create_channel`, `join_channel` lands in the Ed25519-signed audit stream. `verify_chain()` walks it end-to-end.

**You will learn:**

- The `Message` envelope — 13 fields, what each is for, what the service auto-fills
- How `MessagingService` is constructed and what it depends on
- Direct messages (DMs), broadcast channels, role-fanout — three URI schemes, one `send()` call
- Cursor-driven `poll` + `ack`, and how that produces at-least-once delivery without push
- Replay and durability semantics — what's guaranteed when an agent crashes mid-stream
- High-level patterns for consensus on top of pull-based messaging (request-response, voting)
- Failure modes — DLQ entries for oversize bodies, invalid URIs, unauthorized senders, non-member writes

Cross-references: `arcteam/01-team-formation.ipynb` (registry + entities + audit), `arcteam/02-task-distribution.ipynb` (task envelopes, MsgType/Priority semantics — we don't repeat those here), `arcteam/04-team-persistence.ipynb` (NatsBackend, recovery, retention).

## 1. Setup

The boilerplate cell below is the standard Arc walkthrough preamble — locate the repo root, register every package's `src/` (and `tests/` where present), load `.env`. Identical across every Arc notebook so you can skim it once.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Smoke-import the public messaging surface. If any of these fail, fix the environment before continuing — every cell from this point assumes them.

In [ ]:
import hashlib

import arcteam
from arctrust.signer import ED25519, InProcessSigner
from arcteam import (
    AuditLogger,
    Channel,
    Cursor,
    Entity,
    EntityRegistry,
    EntityType,
    MemoryBackend,
    Message,
    MessagingService,
    MsgType,
    Priority,
)

print(f"arcteam version: {arcteam.__version__}")
print(
    "Loaded:",
    [
        MessagingService.__name__,
        Message.__name__,
        Channel.__name__,
        Cursor.__name__,
        MsgType.__name__,
        Priority.__name__,
    ],
)

## 2. Inter-agent messaging primer

Before we touch the API, get the model in your head. `arcteam`'s messaging layer is *pull-based, append-only, URI-addressed*. Three sentences cover almost everything:

1. **Senders write to streams.** A `send()` call appends one record per target URI. Writes are atomic, get a monotonic `seq`, get an Ed25519-signed audit entry.
2. **Recipients pull from streams.** A `poll()` reads forward from the recipient's cursor for that stream. Pull-based means an agent that is not running cannot drop messages — they're waiting when it comes back.
3. **Cursors track progress.** A successful `ack(seq, byte_pos)` advances the cursor. An un-acked message reappears on the next poll. That's how you get at-least-once delivery without push, without delivery confirmations, and without backchannels.

### URI schemes

Four schemes, four routing patterns. The service maps each to a NATS-compatible stream name:

| URI | Stream | Recipient(s) |
|---|---|---|
| `agent://alice` | `arc.agent.alice` | The agent named "alice" — a 1:1 inbox |
| `user://josh` | `arc.agent.josh` | A registered user — same stream shape as agent |
| `channel://ops` | `arc.channel.ops` | Every member of channel `ops` |
| `role://reviewer` | `arc.role.reviewer` | Every entity with `reviewer` in its `roles` list |

Notice `user://` and `agent://` collapse to the same stream prefix (`arc.agent.<name>`). The distinction matters at the registry layer (membership, capabilities) but the wire treats them identically — they both have a 1:1 inbox.

### What this is not

This is not a chat system, not RPC, and not a pub/sub broker with consumer groups. It's the substrate underneath. If your agents need request-response, you build it on top using `thread_id`. If they need voting, you tally polls in your own code. The service stays small on purpose — see `arcteam/CLAUDE.md`'s "core stays under 2,000 LOC" rule.

In [ ]:
# Quick visual: which URI lands in which stream.
from arcteam.types import parse_uri

samples = [
    "agent://alice",
    "user://josh",
    "channel://ops",
    "role://reviewer",
]
print(f"{'URI':<24} {'scheme':<10} {'name'}")
print("-" * 52)
for uri in samples:
    scheme, name = parse_uri(uri)
    # Mirror the messenger's _stream_name_from_uri logic for display.
    if scheme in ("agent", "user"):
        stream = f"arc.agent.{name}"
    else:
        stream = f"arc.{scheme}.{name}"
    print(f"{uri:<24} {scheme:<10} {stream}")

Invalid URIs raise immediately at the type layer — there's no chance of a malformed address slipping into the stream router.

In [ ]:
# this raises — http:// is not in the four-scheme allowlist
try:
    parse_uri("http://not-a-real-scheme")
except ValueError as e:
    print("ValueError:", e)

## 3. `MessagingService` construction

`MessagingService` is the conductor. It wires three collaborators together — a storage backend, an entity registry, and an audit logger — and exposes the `send`/`poll`/`ack`/`get_thread`/channel CRUD surface. It carries no state of its own beyond a small in-memory cursor cache.

| Dependency | Role |
|---|---|
| `StorageBackend` | Where streams, channels, cursors, DLQ entries live (`MemoryBackend` here, `NatsBackend`/JetStream in production) |
| `EntityRegistry` | Used to validate sender registration on every `send()` |
| `AuditLogger` | Ed25519-signed event log; one entry per send, per channel mutation |
| `ui_reporter` (optional) | Duck-typed bridge to `arcui` — fires per routed URI; we don't use it here |

The constructor is plain — no async setup. The `AuditLogger` *does* need `await audit.initialize()` before first use, which loads the chain's last seq/signature if the audit stream already exists.

In [ ]:
# The minimal harness used throughout this notebook.
backend = MemoryBackend()
audit = AuditLogger(backend, signer=InProcessSigner(hashlib.sha256(b"messaging-walkthrough-key").digest(), ED25519))
await audit.initialize()
registry = EntityRegistry(backend, audit)
svc = MessagingService(backend, registry, audit)

print("Backend:        ", type(backend).__name__)
print("Audit ready:    ", audit is not None)
print("Registry ready: ", type(registry).__name__)
print("Service:        ", type(svc).__name__)

Register a small cast of senders and receivers. Three agents, one operator user. Roles are deliberately overlapping (`procurement`, `research`) so we can demonstrate role broadcast later.

In [ ]:
# Build a minimal cast.
async def register_cast(reg: EntityRegistry) -> None:
    await reg.register(
        Entity(
            id="agent://alice",
            name="Alice",
            type=EntityType.AGENT,
            roles=["procurement", "research"],
            capabilities=["search"],
        )
    )
    await reg.register(
        Entity(
            id="agent://bob",
            name="Bob",
            type=EntityType.AGENT,
            roles=["procurement"],
            capabilities=["pricing"],
        )
    )
    await reg.register(
        Entity(
            id="agent://carol",
            name="Carol",
            type=EntityType.AGENT,
            roles=["research"],
            capabilities=["analysis"],
        )
    )
    await reg.register(
        Entity(
            id="user://josh",
            name="Josh",
            type=EntityType.USER,
            roles=["operator"],
        )
    )


await register_cast(registry)
print("Cast:", [e.id for e in await registry.list_entities()])

Every team has at least one channel — the catch-all "general" channel. Channels carry an explicit member list; only members can post into the channel (FR-7, enforced at `send()` time).

In [ ]:
# One channel, three members. Josh is intentionally NOT a member of 'private'.
await svc.create_channel(
    Channel(
        name="alpha",
        description="Project Alpha working room",
        members=["agent://alice", "agent://bob", "agent://carol", "user://josh"],
    )
)
await svc.create_channel(
    Channel(
        name="private",
        description="Restricted: no Josh",
        members=["agent://alice", "agent://bob"],
    )
)

channels = await svc.list_channels()
for ch in channels:
    print(f"  {ch.name:<10}  members={ch.members}")

## 4. Direct messages (1:1)

A direct message is the simplest call. Build a `Message` with a single `agent://` or `user://` URI in `to`, hand it to `svc.send()`. The service:

1. Looks up the sender in the registry. Unknown sender → DLQ + `ValueError`.
2. Validates `body` is ≤ 64 KB. Oversize → DLQ + `ValueError`.
3. Auto-fills `id`, `ts`, and `thread_id` (a new message self-references its thread).
4. Appends one record per `to` URI to the right stream.
5. Writes one `message.sent` audit record per appended target.

The returned `Message` is the same object with auto-fields populated. Reading the recipient's stream gets you the wire form back.

In [ ]:
# Josh DMs Alice. One target → one append, one audit entry.
sent = await svc.send(
    Message(
        sender="user://josh",
        to=["agent://alice"],
        body="Heads up — vendor quotes are in. Please review.",
        msg_type=MsgType.REQUEST,
        priority=Priority.HIGH,
        action_required=True,
    )
)

print("Auto-assigned fields after send():")
print(f"  id:        {sent.id}")
print(f"  ts:        {sent.ts}")
print(f"  seq:       {sent.seq}")
print(f"  thread_id: {sent.thread_id}  (self-referential for new messages)")
print(f"  status:    {sent.status}")
assert sent.thread_id == sent.id, "new messages start their own thread"

Alice polls her inbox. The DM stream for `agent://alice` is `arc.agent.alice` — the registry handle and the stream name share a base. Since Alice has never polled before, her cursor is empty and `poll()` reads from the start.

In [ ]:
# Alice polls her DM inbox.
inbox = await svc.poll("arc.agent.alice", "agent://alice")
for m in inbox:
    print(f"  seq={m.seq}  from={m.sender}  type={m.msg_type.value}  body={m.body!r}")
assert len(inbox) == 1
assert inbox[0].priority == Priority.HIGH

**MsgType + Priority + action_required.** These three fields together describe *what kind of message this is and how urgently the recipient should act.* The wire doesn't enforce any priority semantics — that's recipient policy. But by convention:

- `MsgType.INFO` — purely informational, no response expected.
- `MsgType.REQUEST` — caller wants a reply (`thread_id` chains the response).
- `MsgType.TASK` — directive; recipient owns execution. Covered in `02-task-distribution`.
- `MsgType.RESULT` — task outcome reply.
- `MsgType.ALERT` — `priority=CRITICAL`, jump-the-queue semantics.
- `MsgType.ACK` — short acknowledgement of receipt.

`Priority` is a sort key, not a routing decision — `LOW`/`NORMAL`/`HIGH`/`CRITICAL`. `action_required` is a boolean flag the recipient can use to filter their inbox. Pick conventions for your deployment and stick to them; the service stays out of the way.

In [ ]:
# Two DMs back-to-back, one INFO, one ALERT — same recipient.
await svc.send(
    Message(
        sender="agent://bob",
        to=["agent://alice"],
        body="FYI: pricing API rate limited again.",
        msg_type=MsgType.INFO,
        priority=Priority.LOW,
    )
)
await svc.send(
    Message(
        sender="agent://carol",
        to=["agent://alice"],
        body="Vendor reference check failed — STOP procurement.",
        msg_type=MsgType.ALERT,
        priority=Priority.CRITICAL,
        action_required=True,
    )
)

# Alice's inbox now has 3 messages total. We can sort client-side by priority.
all_msgs = await svc.poll("arc.agent.alice", "agent://alice")
priority_order = {Priority.CRITICAL: 0, Priority.HIGH: 1, Priority.NORMAL: 2, Priority.LOW: 3}
for m in sorted(all_msgs, key=lambda x: priority_order[x.priority]):
    print(f"  [{m.priority.value:<8}] {m.msg_type.value:<8} from={m.sender}: {m.body[:60]}")

**Multi-target sends.** A single `Message` can fan out to multiple URIs. The service treats each as a separate stream append — the recipient on stream A sees `seq=N` for that stream, while stream B has its own seq line. Same `id`, same `body`, same `thread_id` — different `seq` per stream.

In [ ]:
# Bob notifies Alice and Carol with one send().
fan = await svc.send(
    Message(
        sender="agent://bob",
        to=["agent://alice", "agent://carol"],
        body="Both of you: review the quote in #alpha when you have a sec.",
        msg_type=MsgType.REQUEST,
    )
)
print(f"Final seq returned: {fan.seq}  (last stream's seq; alice/carol streams independent)")
print(f"Message id: {fan.id}")

alice_inbox = await svc.poll("arc.agent.alice", "agent://alice")
carol_inbox = await svc.poll("arc.agent.carol", "agent://carol")

# Both inboxes have the same logical message (same id), but each has its own per-stream seq.
alice_match = next(m for m in alice_inbox if m.id == fan.id)
carol_match = next(m for m in carol_inbox if m.id == fan.id)
print(f"  alice sees seq={alice_match.seq}")
print(f"  carol sees seq={carol_match.seq}")
assert alice_match.id == carol_match.id == fan.id

## 5. Channel-based broadcast

A *channel* is a named, member-gated stream. Use channels when:

- Multiple parties need to *see the same conversation* (vs. fanning DMs to everyone).
- You need a stable membership list — joins, leaves, and posts all leave audit footprints.
- You want history accessible to anyone who later joins (the stream is durable; new members `poll(stream, after_seq=0)` to backfill if you want them to).

Member-gated means: only entities whose ID is in `Channel.members` can `send()` *into* the channel. The check is enforced at write time — non-members get a DLQ entry and a `ValueError`. Polling is *not* gated at the messenger layer; if you can name the stream, you can read it. (Production deployments add a separate read-side ACL via `arctrust.policy`.)

In [ ]:
# Alice broadcasts a status to the alpha channel. All four members see it.
await svc.send(
    Message(
        sender="agent://alice",
        to=["channel://alpha"],
        body="Status: 3 of 5 vendors have responded. Will summarize EOD.",
        msg_type=MsgType.INFO,
    )
)

# Each member polls the SAME stream — no per-member fan-out.
for member in ("agent://alice", "agent://bob", "agent://carol", "user://josh"):
    msgs = await svc.poll("arc.channel.alpha", member)
    print(f"  {member:<22} sees {len(msgs)} channel message(s)")

**Role broadcast (`role://`)** is similar in spirit but resolves *at recipient time* via the registry. A `send()` to `role://procurement` writes once to `arc.role.procurement`. Anyone with `procurement` in their `roles` list pulls that stream as part of their `poll_all()` subscription set.

The difference matters: a channel's membership is explicit data on the channel object; a role's membership is whoever's `roles` list contains it at *poll* time. Add a new agent with `roles=["procurement"]` after a role broadcast went out, and they'll see the message on their next poll.

In [ ]:
# Josh broadcasts to all procurement agents.
await svc.send(
    Message(
        sender="user://josh",
        to=["role://procurement"],
        body="Procurement standup at 14:00 UTC. Bring quote summaries.",
        msg_type=MsgType.REQUEST,
    )
)

# Alice (procurement+research) and Bob (procurement) both see it.
for member in ("agent://alice", "agent://bob", "agent://carol"):
    msgs = await svc.poll("arc.role.procurement", member)
    has_role = "procurement" in (await registry.get(member)).roles
    print(f"  {member:<22}  procurement_role={has_role}  visible_messages={len(msgs)}")

**Channel membership enforcement.** Try posting from a non-member and you get a `ValueError` and a DLQ entry. The DLQ (`dlq_list()`) holds every rejected message with a `dlq_reason` explaining why. We come back to the DLQ in §9.

In [ ]:
# this raises — Carol is not in 'private'
try:
    await svc.send(
        Message(
            sender="agent://carol",
            to=["channel://private"],
            body="should not get through",
        )
    )
except ValueError as e:
    print("ValueError:", e)

dlq_after = await svc.dlq_list()
print(f"DLQ entries: {len(dlq_after)}")
print(f"latest reason: {dlq_after[-1]['meta']['dlq_reason']}")

**Channel join/leave** are first-class operations — both audited. Adding a new member doesn't retroactively grant anything, but the new member can `poll(channel_stream, after_seq=0)` to read history. (Or you can refuse them on the read side via your own policy.)

In [ ]:
# Add Carol to 'private' so she can talk to it.
await svc.join_channel("private", "agent://carol")

private_after = next(c for c in await svc.list_channels() if c.name == "private")
print(f"private members now: {private_after.members}")

# Carol's send goes through.
ok = await svc.send(
    Message(
        sender="agent://carol",
        to=["channel://private"],
        body="Joining now. Hi.",
    )
)
print(f"Carol's first private message: seq={ok.seq}")

# Leave is symmetric.
await svc.leave_channel("private", "agent://carol")
private_after = next(c for c in await svc.list_channels() if c.name == "private")
print(f"private members after leave: {private_after.members}")

## 6. Cursor-driven receive

The pull model rests on a single primitive: a `Cursor`. One cursor exists per `(stream, entity_id)` pair. It records:

| Field | Meaning |
|---|---|
| `consumer` | The entity that owns the cursor |
| `stream` | The stream this cursor reads (e.g. `arc.channel.alpha`) |
| `seq` | Last *acked* sequence number — next read returns `seq > this` |
| `byte_pos` | Byte offset into the on-disk JSONL — used by file-backed stores to skip past the consumed prefix without re-scanning; JetStream consumers track this natively via ack floors |
| `updated_at` | ISO-8601 timestamp of the last `ack()` |

`poll()` reads forward from the cursor *without* moving it. `ack(seq, byte_pos)` is the explicit step that advances the cursor. This separation is deliberate — it gives you exactly-once-acked semantics without push and without delivery callbacks. Process the message; if you crash before acking, the next `poll()` returns it again. Process and ack; the cursor moves and you don't see the message again.

The whole thing is **forward-only**. If you call `ack(seq=2)` and then later `ack(seq=1)`, the second call is silently rejected and the cursor stays at seq=2. This protects against bookkeeping bugs in clients (and against malicious replays).

In [ ]:
# Use Bob as our 'lazy reader'. Send 5 messages to alpha, watch the cursor move.
backend2 = MemoryBackend()
audit2 = AuditLogger(backend2, signer=InProcessSigner(hashlib.sha256(b"cursor-demo").digest(), ED25519))
await audit2.initialize()
reg2 = EntityRegistry(backend2, audit2)
svc2 = MessagingService(backend2, reg2, audit2)
await register_cast(reg2)
await svc2.create_channel(
    Channel(
        name="alpha",
        members=["agent://alice", "agent://bob", "agent://carol", "user://josh"],
    )
)

for i in range(5):
    await svc2.send(
        Message(
            sender="agent://alice",
            to=["channel://alpha"],
            body=f"alpha-msg-{i + 1}",
        )
    )

# Bob's first poll: cursor is None, reads from beginning.
batch = await svc2.poll("arc.channel.alpha", "agent://bob", max_messages=10)
print(f"first poll: {[m.body for m in batch]}")
print(f"cursor before any ack: {await svc2.get_cursor('arc.channel.alpha', 'agent://bob')}")

Now Bob acks the first three messages. The cursor jumps to `seq=3`. The next poll returns only `seq > 3`.

In [ ]:
# Bob processes msgs 1-3, acks seq=3.
await svc2.ack("arc.channel.alpha", "agent://bob", seq=3, byte_pos=0)
cur = await svc2.get_cursor("arc.channel.alpha", "agent://bob")
print(
    f"cursor after ack(3): seq={cur.seq}, byte_pos={cur.byte_pos}, updated_at={cur.updated_at[:19]}"
)

remaining = await svc2.poll("arc.channel.alpha", "agent://bob")
print(f"remaining messages: {[m.body for m in remaining]}")
assert [m.seq for m in remaining] == [4, 5]

**Backward acks are rejected.** This is enforced inside `ack()` itself — it fetches the current cursor and refuses to roll back.

In [ ]:
# this is silently rejected — cursor stays at 3
await svc2.ack("arc.channel.alpha", "agent://bob", seq=2, byte_pos=0)
cur = await svc2.get_cursor("arc.channel.alpha", "agent://bob")
print(f"cursor after backward ack attempt: seq={cur.seq}  (unchanged)")
assert cur.seq == 3, "cursor must not move backward"

**`poll_all()` — the entity's full subscription set.** Every entity is implicitly subscribed to:

1. Its own DM inbox (`arc.agent.<name>`).
2. One `arc.role.<role>` stream per role in its registry record.
3. One `arc.channel.<name>` stream per channel it's a member of.

`poll_all(entity_id)` resolves all three sets and polls them in parallel via `asyncio.gather`. The result is a `dict[stream → list[Message]]` containing only streams with new messages.

In [ ]:
# Same harness; produce traffic on three different stream classes.
await svc2.send(Message(sender="agent://alice", to=["agent://carol"], body="DM to carol"))
await svc2.send(Message(sender="user://josh", to=["role://research"], body="research roll-up?"))
await svc2.send(
    Message(sender="agent://alice", to=["channel://alpha"], body="another alpha update")
)

bundle = await svc2.poll_all("agent://carol", max_per_stream=10)
for stream, msgs in bundle.items():
    print(f"  {stream:<28}  count={len(msgs):<3}  preview={[m.body[:30] for m in msgs]}")

`poll_all` is the right primitive for an event loop driving a single agent: in each tick, drain everything new, dispatch by stream type, then ack what you processed. It's also how the dashboard surfaces "who has unread what" without poking into the registry.

In [ ]:
# Subscriptions resolve to a deterministic list per entity.
print(
    "alice subscriptions:",
    svc2.resolve_subscriptions("agent://alice", roles=["procurement", "research"]),
)
print("bob subscriptions:  ", svc2.resolve_subscriptions("agent://bob", roles=["procurement"]))
print("josh subscriptions: ", svc2.resolve_subscriptions("user://josh", roles=["operator"]))

## 7. Replay and durability semantics

Pull-based + append-only means a message persists until *every* entity that needs it has acked it — and even then the data stays on disk. There is no consumer-group accounting; the cursor *is* the consumer state, and there's one cursor per consumer per stream.

Concretely, the storage layer (`MemoryBackend` here, `NatsBackend`/JetStream in production) makes these guarantees:

| Operation | Storage primitive | Guarantee |
|---|---|---|
| `send()` per target | `backend.append_auto_seq` | Atomic seq assignment under lock; no duplicate seqs even with concurrent writers |
| `poll()` | `backend.read_stream(after_seq, byte_pos, limit)` | Returns up to `limit` records with `seq > after_seq` |
| `ack()` | `backend.write` on cursor | Cursor advance is durably acked (JetStream consumer ack floor in `NatsBackend`) |
| Crash mid-process | n/a — no in-flight state | Re-poll returns the un-acked message; nothing is lost |

The same code therefore handles three failure cases identically: a worker crash, a network partition, and a deliberate replay request. All three are "poll without having acked" → message redelivered.

In [ ]:
# Demonstrate redelivery after a 'crash' — agent polled, processed nothing, never acked.
await svc2.send(
    Message(
        sender="agent://alice",
        to=["agent://bob"],
        body="please-process-me",
        msg_type=MsgType.TASK,
    )
)

first_poll = await svc2.poll("arc.agent.bob", "agent://bob")
new_task = next(m for m in first_poll if m.body == "please-process-me")
print(f"first poll seq: {new_task.seq}")

# 'Crash' here — no ack. Subsequent poll redelivers (cursor is whatever it was before).
second_poll = await svc2.poll("arc.agent.bob", "agent://bob")
ids_seen_again = [m.id for m in second_poll if m.id == new_task.id]
print(f"redelivered after 'crash': {bool(ids_seen_again)}")
assert ids_seen_again, "un-acked message must reappear"

**Threaded replay.** `get_thread(stream, thread_id)` walks the *entire* stream history and returns every message tagged with that thread, in `seq` order. This is your primitive for replaying a conversation, regardless of whether the asking agent has acked anything. It's read-only — it does not move any cursor.

In [ ]:
# Build a 3-message conversation in alpha.
root = await svc2.send(
    Message(
        sender="agent://alice",
        to=["channel://alpha"],
        body="Q: which vendor wins on price for >100 units?",
        msg_type=MsgType.REQUEST,
    )
)
await svc2.send(
    Message(
        sender="agent://bob",
        to=["channel://alpha"],
        body="A: vendor C — 8% below B at that volume.",
        thread_id=root.id,
        msg_type=MsgType.RESULT,
    )
)
await svc2.send(
    Message(
        sender="agent://carol",
        to=["channel://alpha"],
        body="Confirming: vendor C also passes our reference checks.",
        thread_id=root.id,
        msg_type=MsgType.INFO,
    )
)

thread = await svc2.get_thread("arc.channel.alpha", root.id)
print(f"Thread {root.id} — {len(thread)} messages:")
for m in thread:
    print(f"  seq={m.seq:<3}  {m.sender:<22}  type={m.msg_type.value:<8}  body={m.body[:60]}")

**Audit chain integrity** — every send, every channel mutation, every cursor cleanup lands in the Ed25519-signed audit stream. `verify_chain()` walks it in batches and returns `(ok, last_verified_seq)`. If the chain breaks at seq N, the call returns `(False, N-1)` and you know exactly where tamper was attempted.

In [ ]:
ok, last = await svc2._audit.verify_chain()
print(f"audit chain valid? {ok}  (verified through audit_seq={last})")
assert ok

records = await backend2.read_stream("audit", "audit", after_seq=0, limit=1000)
event_counts: dict[str, int] = {}
for r in records:
    event_counts[r["event_type"]] = event_counts.get(r["event_type"], 0) + 1
print("audit event distribution:")
for et, n in sorted(event_counts.items(), key=lambda x: -x[1]):
    print(f"  {et:<28} {n}")

**Pointer to `04-team-persistence`.** Everything we've shown sits on `MemoryBackend` — fast, no I/O, throwaway. Production teams swap in `NatsBackend(...)`, which persists the same streams via NATS JetStream and gets crash recovery, durable consumers, and ack-tracked cursors. The messenger code is unchanged. We don't repeat that material here — the next notebook covers it.

## 8. Patterns for consensus (high level)

`arcteam` does not ship a consensus engine. It ships the substrate you build one on. Three patterns recur:

### 8.1 Request–Response

Sender posts a `MsgType.REQUEST`; recipient replies with `MsgType.RESULT` on the same `thread_id`. The original sender polls until they see a `RESULT` matching the thread. This is the simplest 1:1 RPC over messaging.

In [ ]:
# Pattern: request → response, single recipient.
async def request_response(svc, sender, target, prompt, *, deadline_polls=5):
    req = await svc.send(
        Message(
            sender=sender,
            to=[target],
            body=prompt,
            msg_type=MsgType.REQUEST,
        )
    )
    inbox_stream = f"arc.agent.{sender.split('://')[-1]}"
    for _ in range(deadline_polls):
        msgs = await svc.poll(inbox_stream, sender, max_messages=10)
        for m in msgs:
            if m.thread_id == req.id and m.msg_type == MsgType.RESULT:
                return m
    return None


# Mock 'recipient' replies inline by sending RESULT back on the same thread.
async def auto_reply(svc, recipient, requester_thread_id, body):
    await svc.send(
        Message(
            sender=recipient,
            to=[requester_thread_id.split(":replyto:")[0]],
            body=body,
            msg_type=MsgType.RESULT,
            thread_id=requester_thread_id.split(":replyto:")[-1],
        )
    )

Drive the pattern end-to-end. We pre-place a reply so the loop below finishes in one poll. In a real deployment the receiving agent's run loop is what produces the `RESULT`.

In [ ]:
# Alice asks Bob a pricing question; Bob (mocked here) replies on the same thread.
question = await svc2.send(
    Message(
        sender="agent://alice",
        to=["agent://bob"],
        body="best vendor for 250 units?",
        msg_type=MsgType.REQUEST,
    )
)
await svc2.send(
    Message(
        sender="agent://bob",
        to=["agent://alice"],
        body="vendor C, $19.20/unit at 250 vol",
        msg_type=MsgType.RESULT,
        thread_id=question.id,
    )
)

# Alice's loop drains her inbox and picks the reply by thread_id.
alice_inbox = await svc2.poll("arc.agent.alice", "agent://alice", max_messages=20)
reply = next(
    (m for m in alice_inbox if m.thread_id == question.id and m.msg_type == MsgType.RESULT), None
)
print(f"got reply on thread {question.id}: {reply.body if reply else 'NONE'}")

### 8.2 Broadcast → vote → tally

Multi-recipient consensus uses a channel (or `role://`) for the question, then individual `RESULT` messages back. A coordinator counts replies, applies a quorum/majority rule, and decides.

Key points:

- **Quorum** is your business logic — the messenger doesn't know what fraction of recipients you need.
- **Timeouts** are the coordinator's responsibility — keep polling for N seconds, then decide on what you have.
- **Vote integrity** uses signed messages (see `arctrust.sign` / `arctrust.verify`) — the messenger transports signatures in the body or `meta`, but does not verify them itself.

In [ ]:
# Coordinator-driven vote: alice polls workers, tallies a majority.
ballot = await svc2.send(
    Message(
        sender="agent://alice",
        to=["channel://alpha"],
        body="Vote: approve vendor C for the 250-unit order? (yes/no)",
        msg_type=MsgType.REQUEST,
    )
)

# Each voter replies on the alpha channel with thread_id=ballot.id.
for voter, vote in [("agent://bob", "yes"), ("agent://carol", "yes"), ("user://josh", "no")]:
    await svc2.send(
        Message(
            sender=voter,
            to=["channel://alpha"],
            body=vote,
            msg_type=MsgType.RESULT,
            thread_id=ballot.id,
        )
    )

# Tally.
thread = await svc2.get_thread("arc.channel.alpha", ballot.id)
votes = [m.body for m in thread if m.msg_type == MsgType.RESULT]
yes = sum(1 for v in votes if v == "yes")
no = sum(1 for v in votes if v == "no")
print(f"thread {ballot.id} — yes={yes} no={no}  → decision: {'APPROVE' if yes > no else 'REJECT'}")

### 8.3 Leader election (sketch)

For a quorum-based leader election:

1. Every candidate posts an `ALERT` to a known channel with their candidacy + a deterministic priority value.
2. Every candidate polls that channel for `deadline_polls` rounds.
3. After the deadline each candidate independently picks the highest-priority candidacy *they saw* — same input, same output, no coordinator needed.

We sketch it below for two candidates. The "leader" is whoever has the higher priority value; ties broken by ID. Real deployments would sign the messages and verify before counting.

In [ ]:
# Sketch: each candidate posts a candidacy and reads them all back.
import json

candidacy_channel = "alpha"  # reuse existing channel

# Bob and Carol both run for leader.
for who, score in (("agent://bob", 0.8), ("agent://carol", 0.95)):
    await svc2.send(
        Message(
            sender=who,
            to=[f"channel://{candidacy_channel}"],
            body=json.dumps({"candidate": who, "score": score}),
            msg_type=MsgType.ALERT,
            priority=Priority.HIGH,
        )
    )

# Each candidate independently tallies and picks the leader.
all_msgs = await svc2.poll("arc.channel.alpha", "agent://bob", max_messages=1000)
candidacies = []
for m in all_msgs:
    if m.msg_type != MsgType.ALERT:
        continue
    try:
        payload = json.loads(m.body)
        candidacies.append((payload["candidate"], payload["score"]))
    except (json.JSONDecodeError, KeyError):
        continue

candidacies.sort(key=lambda x: (-x[1], x[0]))
leader = candidacies[0][0] if candidacies else None
print(f"observed candidacies: {candidacies}")
print(f"elected leader: {leader}")

**What the messenger does NOT give you for free:**

- *Strong consistency.* If two coordinators run simultaneously, they each see their own poll snapshot. Use a single coordinator or build your own coordination primitive on top.
- *Exactly-once delivery.* It's at-least-once. Idempotent handlers are your responsibility.
- *Cryptographic proof.* The audit chain is a single Ed25519-signed hash chain (`AuditLogger`'s `Signer`), not a signature per individual message. Use `arctrust.sign` to sign the *body* or `meta` if you need non-repudiation.
- *Strict ordering across streams.* Within one stream, `seq` is monotonic. Across streams there's no global order — combining a channel feed and a DM feed needs your own merge logic.

## 9. Failure modes

The service refuses bad sends with a `ValueError` *and* drops the rejected envelope into the Dead Letter Queue (DLQ) before raising. The DLQ is a single append-only stream you can inspect with `dlq_list()`. Each entry has the original `Message` plus `meta.dlq_reason` and `meta.dlq_timestamp`.

| Trigger | `dlq_reason` | Caller-facing exception |
|---|---|---|
| Sender not in registry | `sender_unauthorized` | `ValueError("Sender not registered: ...")` |
| Body > 64 KB | `body_too_large` | `ValueError("Message body exceeds 64KB limit")` |
| URI scheme not in {agent, user, channel, role} | `invalid_address` | `ValueError("Invalid URI: ...")` |
| Sender posts to a channel they're not a member of | `not_channel_member` | `ValueError("Sender ... is not a member of channel://...")` |

All four are tested below.

In [ ]:
# Fresh harness for the failure tour — keeps the DLQ list clean.
fail_backend = MemoryBackend()
fail_audit = AuditLogger(fail_backend, signer=InProcessSigner(hashlib.sha256(b"fail-demo").digest(), ED25519))
await fail_audit.initialize()
fail_reg = EntityRegistry(fail_backend, fail_audit)
fail_svc = MessagingService(fail_backend, fail_reg, fail_audit)

await register_cast(fail_reg)
await fail_svc.create_channel(
    Channel(
        name="alpha",
        members=["agent://alice", "agent://bob"],
    )
)
print("harness ready")

In [ ]:
# Failure 1: unregistered sender.
try:
    await fail_svc.send(
        Message(
            sender="agent://ghost",
            to=["channel://alpha"],
            body="hello",
        )
    )
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Failure 2: oversize body. Bypass Pydantic validation with model_construct
# to mimic a programmatic upstream that didn't pre-validate.
big_body = "x" * (65536 + 1000)
oversize = Message.model_construct(
    seq=0,
    id="",
    ts="",
    sender="agent://alice",
    to=["channel://alpha"],
    thread_id=None,
    msg_type=MsgType.INFO,
    priority=Priority.NORMAL,
    action_required=False,
    body=big_body,
    refs=[],
    status="sent",
    meta={},
)
try:
    await fail_svc.send(oversize)
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Failure 3: malformed URI scheme.
try:
    await fail_svc.send(
        Message(
            sender="agent://alice",
            to=["http://not-allowed"],
            body="hello",
        )
    )
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Failure 4: non-member channel post.
try:
    await fail_svc.send(
        Message(
            sender="agent://carol",  # not in alpha members list
            to=["channel://alpha"],
            body="should fail",
        )
    )
except ValueError as e:
    print("ValueError:", e)

Now inspect the DLQ. Every rejected message is there with its reason. Production runs hook this stream into an alert path — it's the closest thing the messenger has to a "something went wrong" signal.

In [ ]:
dlq = await fail_svc.dlq_list()
print(f"DLQ entries: {len(dlq)}")
for entry in dlq:
    reason = entry["meta"]["dlq_reason"]
    sender = entry["sender"]
    targets = entry["to"]
    print(f"  reason={reason:<22} sender={sender:<20} to={targets}")

**Stale-cursor cleanup** is its own failure-prevention surface. A long-lived deployment accumulates cursors for entities that no longer exist (deleted agents, expired pairings). `cleanup_stale_cursors(max_age_hours)` walks every cursor and removes anyone whose `updated_at` is older than the threshold. The action emits a `cursor.cleanup` audit event.

In [ ]:
# Force-create a cursor by acking, then sweep with a 0-hour threshold to remove it.
await fail_svc.ack("arc.channel.alpha", "agent://alice", seq=1, byte_pos=0)
removed = await fail_svc.cleanup_stale_cursors(max_age_hours=0)
print(f"cursors removed: {removed}")

# A cleanup that finds nothing returns 0 and emits no audit.
removed_again = await fail_svc.cleanup_stale_cursors(max_age_hours=0)
print(f"second sweep (already clean): {removed_again}")

**Things the messenger explicitly does NOT do** — and where to look instead:

- *Encryption at rest / in transit.* Backend's responsibility. `NatsBackend` writes plaintext JetStream records; deploy NATS with TLS/at-rest encryption or wrap it.
- *Per-message authorization.* Use `arctrust.policy.PolicyPipeline`. The messenger only enforces channel membership.
- *Deduplication on send.* Caller's job. Re-sending the same logical message *will* produce two distinct stream entries with different `seq`s and `id`s.
- *Backpressure.* No queue depth, no rejection on overload. The backend is sized so it never overflows; if you need backpressure, build it at the runtime layer above (`arcrun` queue module) or in front of the registry.

## 10. Summary

**What you saw in this notebook:**

- The wire model: `MessagingService` is pull-based, append-only, URI-addressed. Senders write to streams; recipients pull; cursors track per-consumer progress.
- Four URI schemes: `agent://`, `user://`, `channel://`, `role://`. They map deterministically to NATS-shaped stream names (`arc.agent.<n>`, `arc.channel.<n>`, `arc.role.<n>`).
- The 13-field `Message` envelope and what the service auto-fills (`id`, `ts`, `seq`, `thread_id`, `status`).
- DMs (1:1), channel broadcast (member-gated), and role broadcast (resolved at recipient time via the registry).
- `Cursor`-driven receive: `poll()` reads forward, `ack()` advances, backward acks are silently rejected. `poll_all()` drains DM + channels + roles in parallel.
- Replay and durability: un-acked messages reappear on next poll. `get_thread()` walks the full stream history. The Ed25519-signed audit chain is verifiable end-to-end via `verify_chain()`.
- High-level consensus patterns: request–response on `thread_id`, broadcast-vote-tally over channels, leader election as a deterministic post-and-tally on a known channel. The messenger gives you the substrate; quorum and signing live above it.
- Failure modes route through a DLQ with an explicit `dlq_reason`. Stale-cursor cleanup and channel membership enforcement keep long-running deployments healthy.

**Public API surface covered:**

- `arcteam.MessagingService` — `send`, `poll`, `poll_all`, `ack`, `get_cursor`, `cleanup_stale_cursors`, `create_channel`, `join_channel`, `leave_channel`, `list_channels`, `get_thread`, `dlq_list`, `resolve_subscriptions`
- `arcteam.Message` — full envelope, auto-assigned fields, body-size validation
- `arcteam.Channel` — name, description, members
- `arcteam.Cursor` — consumer/stream/seq/byte_pos/updated_at
- `arcteam.MsgType` — INFO, REQUEST, TASK, RESULT, ALERT, ACK
- `arcteam.Priority` — LOW, NORMAL, HIGH, CRITICAL
- `arcteam.types.parse_uri` — URI scheme validator (used internally)
- Reuse from `01-team-formation`: `EntityRegistry`, `Entity`, `EntityType`, `MemoryBackend`, `AuditLogger`

**Where this fits in the four pillars:**

- **Identity** — every send is gated on a registry lookup of `sender`. Unknown sender → DLQ.
- **Sign** — message bodies are *transported* unsigned by the messenger; non-repudiation is added by the caller via `arctrust.sign` over `body` or `meta`.
- **Authorize** — channel membership is enforced inside `send()`. Anything richer (per-message ACL, classification labels) goes through `arctrust.policy.PolicyPipeline`.
- **Audit** — every send, channel mutation, and cursor cleanup lands in the Ed25519-signed audit stream. `verify_chain()` is the tamper detector.

**Next:**

- `arcteam/02-task-distribution.ipynb` — `MsgType.TASK`, `Priority` semantics, fan-out patterns for assigning work across the roster.
- `arcteam/04-team-persistence.ipynb` — `NatsBackend`, `TeamMemoryService`, `TeamFileStore`, durable storage and crash recovery for everything we just built on `MemoryBackend`.